# BT02: LSB Steganography on Palette-Based GIF Images

**Method:** Fridrich's LSB embedding on palette-indexed (GIF) images

**Reference:** Slide deck `05-DataHiding on Image(P2)`

---
**Name:** Lê Thị Hồng Hạnh

**Student ID:** 22127103

---
## Background

### GIF and Palette-Based Images
GIF images use a **color palette** (also called a colormap or lookup table) of up to 256 colors. Each pixel stores an **index** into this palette rather than a direct RGB value. This structure enables a unique steganographic approach: instead of modifying pixel intensities directly, we modify the palette or remap pixel indices.

### Fridrich's LSB Method for GIF
The key insight is: for each palette color `(R, G, B)`, compute `parity = (R + G + B) % 2`. The message bit is encoded by ensuring the palette index pointed to by each pixel has a color whose parity matches the desired bit. If it does not match, the pixel index is **swapped** to the nearest palette color with the opposite parity.

This keeps the palette unchanged — only the pixel index map is altered, making changes invisible if the two nearest-parity-pair colors are visually similar.

### Bit Representation Convention
- `tobits(s)` converts a string to a **bit string** (e.g., `'A'` → `'01000001'`)
- `frombits(bs)` converts a bit string back to a string
- All embedding and extraction operates on bit strings for clarity and efficiency

---
## Imports

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import hashlib
import struct

---
## Part 1: Bit String Utilities

These helper functions convert between text strings and **bit strings** (strings of `'0'` and `'1'` characters). Using a bit string (rather than a list of integers) allows efficient string slicing, concatenation, and direct use with Python's `int(bits, 2)` for parsing.

In [ ]:
def text_to_bits(text: str) -> str:
    """
    Convert a text string to a binary string using UTF-8 encoding.
    
    Args:
        text: The text string to convert
    
    Returns:
        Binary string (e.g., "Hi" → "0100100001101001")
    
    Example:
        >>> text_to_bits('A')
        '01000001'
        >>> text_to_bits('Hi')
        '0100100001101001'
    """
    if not text:
        return ''
    
    # Encode text to UTF-8 bytes
    text_bytes = text.encode('utf-8')
    
    # Convert each byte to 8-bit binary string
    bits = ''.join(format(byte, '08b') for byte in text_bytes)
    
    return bits


def bits_to_text(bits: str) -> str:
    """
    Convert a binary string back to text using UTF-8 encoding.
    
    Args:
        bits: Binary string (must be valid UTF-8 when decoded)
    
    Returns:
        Decoded text string
    
    Example:
        >>> bits_to_text('01000001')
        'A'
        >>> bits_to_text('0100100001101001')
        'Hi'
    """
    bits = bits[:len(bits) // 8 * 8]
    if not bits:
        return ''
    try:        
        return bytes(int(bits[i:i+8], 2) for i in range(0, len(bits), 8)).decode('utf-8')
    except:
        return ''

# --- Sanity checks ---
assert text_to_bits('A') == '01000001'
assert text_to_bits('Hi') == '0100100001101001'
assert bits_to_text(text_to_bits('Hello, World!')) == 'Hello, World!'
print("tobits/frombits: OK")
print(f"  tobits('A') = '{text_to_bits('A')}'")
print(f"  frombits('01000001') = '{bits_to_text('01000001')}'")

---
## Part 2: Palette Analysis & Visualization

### 2a. Palette Parity Visualization

Before embedding, it is useful to visualize the palette and understand how many colors have even vs odd parity (`(R+G+B) % 2`). Embedding capacity depends on having both parity groups well-represented.

In [ ]:
def visualize_palette(img_file: str, title: str = None):
    """Display the color palette of a GIF image, color-coded by parity.
    
    Colors with even parity (R+G+B) % 2 == 0 are marked with a white border.
    Colors with odd parity are marked with a black border.
    
    Args:
        img_file (str): Path to GIF image.
        title (str): Optional plot title.
    """
    img = Image.open(img_file)
    raw_palette = img.getpalette()  # flat list: [R0,G0,B0, R1,G1,B1, ...]
    palette = np.array(raw_palette, dtype=np.uint8).reshape(-1, 3)  # (256, 3)
    
    parities = (palette[:, 0].astype(int) + palette[:, 1].astype(int) + palette[:, 2].astype(int)) % 2
    n_even = np.sum(parities == 0)
    n_odd  = np.sum(parities == 1)
    
    # Display as a 16x16 grid
    grid = palette.reshape(16, 16, 3)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Left: raw palette
    axes[0].imshow(grid, aspect='auto')
    axes[0].set_title(f'Full Palette (16×16)\n{title or img_file}')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    
    # Right: parity map (even=light, odd=dark)
    parity_grid = parities.reshape(16, 16)
    axes[1].imshow(parity_grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    axes[1].set_title(f'Parity Map\nEven (green)={n_even}  Odd (red)={n_odd}')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    even_patch = mpatches.Patch(color='green', label=f'Even parity ({n_even})')
    odd_patch  = mpatches.Patch(color='red',   label=f'Odd parity ({n_odd})')
    axes[1].legend(handles=[even_patch, odd_patch], loc='lower right', fontsize=8)
    
    plt.tight_layout()
    plt.show()
    return palette, parities


# Uncomment to visualize (requires lena.gif to be present):
palette, parities = visualize_palette('lena.gif', 'lena.gif')

### 2b. Capacity Analysis

**Embedding capacity** in Fridrich's method depends on:
- The number of pixels in the image
- Whether each pixel's palette color can be swapped (i.e., a color of opposite parity exists in the palette)

In practice, if *all* palette entries have a nearest neighbor of opposite parity, capacity = number of pixels bits = ⌊pixels / 8⌋ characters.

In [ ]:
def analyze_capacity(img_file: str) -> dict:
    """Analyze the steganographic capacity of a GIF image.
    
    For Fridrich's method, each pixel can carry 1 bit if its palette color
    has at least one neighbor of opposite parity in the palette.
    
    Args:
        img_file (str): Path to GIF image.
    
    Returns:
        dict: Capacity statistics including bits, bytes, characters, and parity info.
    """
    img = Image.open(img_file)
    pixels = np.array(img)                     # index map, shape (H, W)
    raw_palette = img.getpalette()
    palette = np.array(raw_palette, dtype=np.uint8).reshape(-1, 3)
    
    # Compute parity per palette entry
    parities = (palette[:, 0].astype(int) +
                palette[:, 1].astype(int) +
                palette[:, 2].astype(int)) % 2
    
    n_even = int(np.sum(parities == 0))
    n_odd  = int(np.sum(parities == 1))
    total_pixels = pixels.size

    # A pixel is "embeddable" if its palette entry has a partner of opposite parity
    # (i.e., both parity groups are non-empty — which is almost always true)
    can_embed = (n_even > 0 and n_odd > 0)
    
    capacity_bits  = total_pixels if can_embed else 0
    capacity_bytes = capacity_bits // 8
    
    # Average color distance introduced (estimated from palette)
    avg_dist = compute_avg_swap_distance(palette, parities)
    
    stats = {
        'image': img_file,
        'size': f'{img.width}×{img.height}',
        'total_pixels': total_pixels,
        'palette_size': len(palette),
        'n_even_parity': n_even,
        'n_odd_parity': n_odd,
        'capacity_bits': capacity_bits,
        'capacity_bytes': capacity_bytes,
        'capacity_chars': capacity_bytes,
        'avg_swap_distance': avg_dist,
    }
    
    print(f"=== Capacity Analysis: {img_file} ===")
    for k, v in stats.items():
        if k != 'image':
            print(f"  {k:25s}: {v}")
    return stats


def compute_avg_swap_distance(palette: np.ndarray, parities: np.ndarray) -> float:
    """Compute the average Euclidean color distance when swapping to the nearest
    opposite-parity palette entry.
    
    This is a proxy for visual distortion: lower = less visible steganography.
    
    Args:
        palette (np.ndarray): Shape (N, 3) array of RGB palette colors.
        parities (np.ndarray): Shape (N,) array of parity values (0 or 1).
    
    Returns:
        float: Mean swap distance across all palette entries.
    """
    palette_f = palette.astype(float)
    distances = []
    for i in range(len(palette)):
        target_parity = 1 - parities[i]
        candidates = np.where(parities == target_parity)[0]
        if len(candidates) == 0:
            continue
        diffs = palette_f[candidates] - palette_f[i]
        dists = np.sqrt(np.sum(diffs ** 2, axis=1))
        distances.append(np.min(dists))
    return float(np.mean(distances)) if distances else 0.0

analyze_capacity('lena.gif')
analyze_capacity('baboon.gif')

---
## Part 3: Embedding Function (Fridrich's Method)

**Algorithm overview:**
1. Read the message file and convert to a bit string.
2. Prepend a 32-bit length header so the extractor knows when to stop.
3. For each pixel (in raster order), read the corresponding palette color's parity.
4. If parity matches the next message bit → keep the pixel index unchanged.
5. If parity does not match → find the nearest palette color of opposite parity and remap this pixel to that index.
6. Save the modified pixel index map with the original palette.

> ⚠️ **Note on data types:** When computing color distances, cast `uint8` values to `int` (or `float`) before subtraction to avoid unsigned underflow.
>
> ⚠️ **Tie-breaking:** When multiple palette entries are equally close, always pick the one with the **smallest index** (use `kind='stable'` in any sort).

In [ ]:
def _palette_info(img):
    """Return (pal_int, parities, nearest_partner) for an open PIL GIF image."""
    pal_int  = np.array(img.getpalette(), dtype=np.uint8).reshape(-1, 3).astype(int)
    parities = (pal_int[:, 0] + pal_int[:, 1] + pal_int[:, 2]) % 2
    N        = len(pal_int)
    nearest  = np.zeros(N, dtype=int)
    for i in range(N):
        target    = 1 - parities[i]
        cands     = np.where(parities == target)[0]
        diffs     = pal_int[cands] - pal_int[i]
        dists     = np.sum(diffs ** 2, axis=1)
        nearest[i] = cands[np.argsort(dists, kind='stable')[0]]
    return pal_int, parities, nearest

In [ ]:
def embed(msg_file: str, cover_img_file: str, stego_img_file: str) -> bool:
    """Embed a secret message into a GIF image using Fridrich's LSB method.
    
    The message is preceded by a 32-bit (4-byte) length header indicating
    the number of message characters, allowing self-terminating extraction.
    
    Each pixel encodes 1 bit: the palette index is remapped to the nearest
    palette color whose (R+G+B) parity matches the desired message bit.
    
    Args:
        msg_file (str): Path to the plaintext message file.
        cover_img_file (str): Path to the cover GIF image.
        stego_img_file (str): Output path for the stego GIF image.
    
    Returns:
        bool: True if embedding succeeded, False if the image has insufficient capacity.
    """
    # Hint: use _palette_info(img) to get (pal_int, parities, nearest_partner) arrays
    # YOUR CODE HERE
    with open(msg_file, 'r') as f:
        msg_str = f.read()
    msg_bits = text_to_bits(msg_str)
    
    img = Image.open(cover_img_file)
    pixels = np.array(img)
    
    # Check capacity
    if len(msg_bits) + 32 > pixels.size:
        return False
    
    # Bit string to be embedded = 32 bit length + msg_bits
    msg_bytes_len = len(msg_str.encode('utf-8'))
    embedded_bits = f'{msg_bytes_len:032b}' + msg_bits
    _, parities, nearest_partner = _palette_info(img)
    
    flat_pixels = pixels.flatten()
    for b in range(len(embedded_bits)):
        i = flat_pixels[b]
        if int(embedded_bits[b]) != parities[i]:
            flat_pixels[b] = nearest_partner[i]
            
    stego_pixels = flat_pixels.reshape(pixels.shape)
            
    stego_img = Image.fromarray(stego_pixels.astype(np.uint8), mode='P')
    stego_img.putpalette(img.getpalette())
    stego_img.save(stego_img_file)
    
    return True

In [ ]:
# TEST 1: Message too large — should return False
result = embed('msg2.txt', 'lena.gif', 'lena_stego.gif')
assert result == False, "Expected False when message exceeds capacity"
print("TEST 1 passed: overflow detected correctly")

In [ ]:
# TEST 2: Normal embed into lena.gif
result = embed('msg.txt', 'lena.gif', 'lena_stego.gif')
assert result == True
stego_img           = Image.open('lena_stego.gif')
stego_pixels        = np.array(stego_img)
stego_palette       = stego_img.getpalette()
correct_stego_img   = Image.open('correct_lena_stego.gif')
correct_stego_pixels  = np.array(correct_stego_img)
correct_stego_palette = correct_stego_img.getpalette()
assert np.all(stego_pixels == correct_stego_pixels)
assert stego_palette == correct_stego_palette
print("TEST 2 passed: lena.gif embedding correct")

In [ ]:
# TEST 3: Embed into baboon.gif
result = embed('msg.txt', 'baboon.gif', 'baboon_stego.gif')
assert result == True
stego_img           = Image.open('baboon_stego.gif')
stego_pixels        = np.array(stego_img)
stego_palette       = stego_img.getpalette()
correct_stego_img   = Image.open('correct_baboon_stego.gif')
correct_stego_pixels  = np.array(correct_stego_img)
correct_stego_palette = correct_stego_img.getpalette()
assert np.all(stego_pixels == correct_stego_pixels)
assert stego_palette == correct_stego_palette
print("TEST 3 passed: baboon.gif embedding correct")

### Reflection: Visual Invisibility Comparison

Compare the visual quality of embedding on `lena.gif` vs `baboon.gif`. Why might one image show more visible distortion than the other?

**Hint:** Use `compute_avg_swap_distance()` on both images' palettes and compare the values. Also consider the textural complexity of each image.

YOUR ANSWER HERE

The `lena.gif` image shows significantly more visible distortion and noise compared to `baboon.gif`
- **Average swap distance**: The palette of `lena.gif` (`avg_swap_distance` $\approx$ 20.85) typically has a larger average swap distance than `baboon.gif` has (`avg_swap_distance` $\approx$ 14.03). This means that when the algorithm is forced to change a pixel's color to hide a bit, it must "jump" to a replacement color that is further away in the RGB color space, leading to more color deviations.
- **Textural complexity**: `lena.gif` contains many smooth, flat areas with low textural complexity (such as the skin tones and the background) and the human eye is sensitive to unusual speckled details on a smooth surface. In contrast, `baboon.gif` is characterized by high-frequency details and extreme textural complexity (the fine fur, whiskers, and wrinkles), so the noise generated by color swapping blends in and is effectively masked by the image's natural color variations and the human eye is less sensitive to noise in highly textured regions.

---
## Part 4: Extraction Function

Extraction reads the parity of each pixel's palette color in raster order. The first 32 bits are decoded as the message length; subsequent bits (up to `length × 8`) are decoded as the message text.

In [ ]:
def extract(stego_img_file: str, extr_msg_file: str) -> str:
    """Extract a secret message from a stego GIF image.
    
    Reads pixel palette parities in raster order. The first 32 bits encode
    the message length (number of characters); the next length×8 bits are
    the message.
    
    Args:
        stego_img_file (str): Path to the stego GIF image.
        extr_msg_file (str): Output path for the extracted message.
    
    Returns:
        str: The extracted message string.
    """
    # YOUR CODE HERE
    stego_img = Image.open(stego_img_file)
    _, parities, _ = _palette_info(stego_img)
    stego_flat_pixels = np.array(stego_img).flatten()
    
    length_str = ''.join([str(parities[i]) for i in stego_flat_pixels[:32]])
    length = int(length_str, 2) * 8
    
    msg_bits = ''.join([str(parities[i]) for i in stego_flat_pixels[32:32+length]])
    extr_msg = bits_to_text(msg_bits)
    
    with open(extr_msg_file, 'w') as f:
        f.write(extr_msg)
        
    return extr_msg

In [ ]:
# TEST: Extract from baboon stego and compare
extract('correct_baboon_stego.gif', 'extr_msg.txt')
with open('extr_msg.txt', 'r') as f:
    extr_msg = f.read()
with open('msg.txt', 'r') as f:
    correct_extr_msg = f.read()
assert extr_msg == correct_extr_msg
print("Extraction test passed!")

---
## Part 5: Visual Comparison of Cover vs Stego Image

Run the cell below to render both the cover and stego images side-by-side, along with a difference map highlighting which pixels were altered.

In [ ]:
def compare_cover_stego(cover_file: str, stego_file: str):
    """Side-by-side comparison of cover and stego images, with difference map,
    MSE, and PSNR.

    Args:
        cover_file (str): Path to cover GIF.
        stego_file (str): Path to stego GIF.
    """
    cover = Image.open(cover_file)
    stego = Image.open(stego_file)

    cover_px = np.array(cover)
    stego_px = np.array(stego)

    # Convert palette indices to RGB for display and metric computation
    cover_rgb = np.array(cover.convert('RGB')).astype(float)
    stego_rgb = np.array(stego.convert('RGB')).astype(float)

    # --- Pixel change stats (on palette index map) ---
    diff_mask   = (cover_px != stego_px)
    diff_count  = int(np.sum(diff_mask))
    pct_changed = 100.0 * diff_count / cover_px.size

    # --- MSE and PSNR (on RGB values) ---
    mse = np.mean((cover_rgb - stego_rgb) ** 2)
    if mse == 0:
        psnr = float('inf')
    else:
        psnr = 10 * np.log10((255.0 ** 2) / mse)

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(cover_rgb.astype(np.uint8))
    axes[0].set_title(f'Cover Image\n{cover_file}')
    axes[0].axis('off')

    axes[1].imshow(stego_rgb.astype(np.uint8))
    axes[1].set_title(f'Stego Image\n{stego_file}')
    axes[1].axis('off')

    axes[2].imshow(diff_mask, cmap='hot')
    axes[2].set_title(f'Changed Pixels (white)\n{diff_count} pixels ({pct_changed:.1f}%)')
    axes[2].axis('off')

    fig.suptitle(f'MSE = {mse:.4f}    PSNR = {psnr:.2f} dB', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Pixels changed : {diff_count} / {cover_px.size} ({pct_changed:.2f}%)")
    print(f"MSE            : {mse:.4f}")
    print(f"PSNR           : {psnr:.2f} dB")

In [ ]:
# NOTE: Run this cell only after completing Parts 3 and 4 (embed + extract).
# lena_stego.gif and baboon_stego.gif must exist before these calls will work.
compare_cover_stego('lena.gif', 'lena_stego.gif')
compare_cover_stego('baboon.gif', 'baboon_stego.gif')

---
## Part 6: Secure Embedding with Password-Based Pixel Shuffling

In standard Fridrich embedding, bits are written into pixels in sequential raster order. An attacker who knows the method can trivially extract the message. A simple improvement is to **shuffle the pixel order** using a pseudo-random sequence seeded by a password hash. This does not change the per-pixel encoding mechanism, only the order in which pixels are used.

**Security note:** This is *security through obscurity* — it adds a shared-secret requirement but is not cryptographically strong. For real applications, encrypt the message before embedding.

In [ ]:
import hashlib

def embed_secure(msg_file: str, cover_img_file: str, stego_img_file: str, password: str) -> bool:
    """Embed a secret message using Fridrich's method with password-based pixel shuffling.
    
    Pixels are visited in a pseudo-random order determined by a seed derived from
    the SHA-256 hash of the password. This makes sequential extraction impossible
    without knowing the password.
    
    Args:
        msg_file (str): Path to plaintext message file.
        cover_img_file (str): Path to cover GIF image.
        stego_img_file (str): Output path for stego GIF image.
        password (str): Shared secret used to generate the pixel order.
    
    Returns:
        bool: True on success, False if capacity exceeded.
    """
    # YOUR CODE HERE
    with open(msg_file, 'r') as f:
        msg_str = f.read()
    msg_bits = text_to_bits(msg_str)
    
    img = Image.open(cover_img_file)
    pixels = np.array(img)
    
    # Check capacity
    if len(msg_bits) + 32 > pixels.size:
        return False
    
    # Bit string to be embedded = 32 bit length + msg_bits
    msg_bytes_len = len(msg_str.encode('utf-8'))
    embedded_bits = f'{msg_bytes_len:032b}' + msg_bits
    
    # Maximum value of np.random.seed() argument is 2^32
    seed = int(hashlib.sha256(password.encode()).hexdigest(), 16) % (2**32)
    np.random.seed(seed)
    perm = np.random.permutation(pixels.size)
    
    _, parities, nearest_partner = _palette_info(img)
    
    flat_pixels = pixels.flatten()
    for b in range(len(embedded_bits)):
        pos = perm[b]
        i = flat_pixels[pos]
        if int(embedded_bits[b]) != parities[i]:
            flat_pixels[pos] = nearest_partner[i]
            
    pixels = flat_pixels.reshape(pixels.shape)
            
    stego_img = Image.fromarray(pixels.astype(np.uint8), mode='P')
    stego_img.putpalette(img.getpalette())
    stego_img.save(stego_img_file)
    
    return True

In [ ]:
def extract_secure(stego_img_file: str, extr_msg_file: str, password: str) -> str:
    """Extract a secret message embedded with embed_secure().
    
    Args:
        stego_img_file (str): Path to stego GIF image.
        extr_msg_file (str): Output path for extracted message.
        password (str): The same password used during embedding.
    
    Returns:
        str: The extracted message string.
    """
    # YOUR CODE HERE   
    stego_img = Image.open(stego_img_file)
    _, parities, _ = _palette_info(stego_img)
    stego_flat_pixels = np.array(stego_img).flatten()
    
    seed = int(hashlib.sha256(password.encode()).hexdigest(), 16) % (2**32)
    np.random.seed(seed)
    perm = np.random.permutation(stego_flat_pixels.size)
    
    length_str = ''.join([str(parities[stego_flat_pixels[pos]]) for pos in perm[:32]])
    length = int(length_str, 2) * 8
    
    msg_bits = ''.join([str(parities[stego_flat_pixels[pos]]) for pos in perm[32:32+length]])
    extr_msg = bits_to_text(msg_bits)
    with open(extr_msg_file, 'w') as f:
        f.write(extr_msg)
        
    return extr_msg

In [ ]:
# TEST: Secure embed/extract round-trip
result = embed_secure('msg.txt', 'lena.gif', 'lena_secure_stego.gif', password='secret123')
assert result == True
extracted = extract_secure('lena_secure_stego.gif', 'extr_secure_msg.txt', password='secret123')
with open('msg.txt', 'r') as f:
    original = f.read()
assert extracted == original
print("Secure embed/extract round-trip: PASSED")

# Wrong password should produce garbage
extracted_wrong = extract_secure('lena_secure_stego.gif', 'extr_wrong.txt', password='wrong')
assert extracted_wrong != original
print("Wrong password produces different output: PASSED")

---
## Part 7 (Extended — Optional Bonus): Multi-Bit LSB (2-Bit Mode)

> **This part is optional.** Complete Parts 3–6 first. Bonus marks are awarded for a correct implementation.

Instead of encoding 1 bit per pixel, we can encode **2 bits per pixel** by examining the **2 least significant bits** of `(R+G+B)`. This doubles the capacity but increases the color distance introduced per swap.

The 2-bit parity of a palette color is `(R + G + B) % 4`, which gives values `{0, 1, 2, 3}`. Each pixel now encodes a 2-bit symbol.

In [ ]:
def _palette_info_2bit(img):
    """Return (pal_int, parities, nearest_partner) for an open PIL GIF image."""
    pal_int  = np.array(img.getpalette(), dtype=np.uint8).reshape(-1, 3).astype(int)
    parities = (pal_int[:, 0] + pal_int[:, 1] + pal_int[:, 2]) % 4
    N        = len(pal_int)
    nearest  = np.zeros((N,4), dtype=int)
    for i in range(N):
        for target in range(4):
            if parities[i] == target:
                nearest[i][target] = i
                continue
            cands     = np.where(parities == target)[0]
            diffs     = pal_int[cands] - pal_int[i]
            dists     = np.sum(diffs ** 2, axis=1)
            nearest[i][target] = cands[np.argsort(dists, kind='stable')[0]]
    return pal_int, parities, nearest

def embed_2bit(msg_file: str, cover_img_file: str, stego_img_file: str) -> bool:
    """Embed a secret message using 2-bit-per-pixel palette parity.
    
    Instead of encoding 1 bit per pixel (using parity mod 2), this function
    encodes 2 bits per pixel by targeting the value (R+G+B) % 4 for each
    palette entry. The pixel index is remapped to the nearest palette color
    with the matching 2-bit symbol.
    
    Args:
        msg_file (str): Path to plaintext message file.
        cover_img_file (str): Path to cover GIF image.
        stego_img_file (str): Output path for stego GIF.
    
    Returns:
        bool: True on success, False if capacity exceeded.
    
    Notes:
        - Capacity doubles compared to 1-bit mode: ⌊pixels / 4⌋ characters.
        - Distortion increases since swaps may be to more distant palette entries.
        - The length header is still 32 bits (16 pixels in 2-bit mode).
    """
    # YOUR CODE HERE
    with open(msg_file, 'r') as f:
        msg_str = f.read()
    msg_bits = text_to_bits(msg_str)
    
    img = Image.open(cover_img_file)
    pixels = np.array(img)
    
    # Check capacity
    if len(msg_bits) + 32 > pixels.size * 2:
        return False
    
    # Bit string to be embedded = 32 bit length + msg_bits
    msg_bytes_len = len(msg_str.encode('utf-8'))
    embedded_bits = f'{msg_bytes_len:0{32}b}' + msg_bits    
    _, parities, nearest_partner = _palette_info_2bit(img)
    
    flat_pixels = pixels.flatten()
    for b in range(0, len(embedded_bits), 2):
        i = flat_pixels[b // 2]
        if int(embedded_bits[b:b+2], 2) != parities[i]:
            flat_pixels[b // 2] = nearest_partner[i][int(embedded_bits[b:b+2], 2)]
            
    pixels = flat_pixels.reshape(pixels.shape)
            
    stego_img = Image.fromarray(pixels.astype(np.uint8), mode='P')
    stego_img.putpalette(img.getpalette())
    stego_img.save(stego_img_file)
    
    return True

In [ ]:
def extract_2bit(stego_img_file: str, extr_msg_file: str) -> str:
    """Extract a secret message embedded with embed_2bit().
    
    Args:
        stego_img_file (str): Path to stego GIF.
        extr_msg_file (str): Output path for extracted message.
    
    Returns:
        str: Extracted message string.
    """
    # YOUR CODE HERE    
    stego_img = Image.open(stego_img_file)
    _, parities, _ = _palette_info_2bit(stego_img)
    stego_flat_pixels = np.array(stego_img).flatten()
    
    length_str = ''.join([f'{parities[i]:0{2}b}' for i in stego_flat_pixels[:16]])
    length = int(length_str, 2) * 8
    
    msg_bits = ''.join([f'{parities[i]:0{2}b}' for i in stego_flat_pixels[16:16+(length //2)]])
    extr_msg = bits_to_text(msg_bits)
    with open(extr_msg_file, 'w') as f:
        f.write(extr_msg)
        
    return extr_msg

In [ ]:
# TEST: 2-bit mode round-trip
result = embed_2bit('msg.txt', 'lena.gif', 'lena_stego_2bit.gif')
assert result == True
extracted = extract_2bit('lena_stego_2bit.gif', 'extr_msg_2bit.txt')
with open('msg.txt', 'r') as f:
    original = f.read()
assert extracted == original
print("2-bit mode round-trip: PASSED")

# Compare visual distortion: 1-bit vs 2-bit
compare_cover_stego('lena.gif', 'lena_stego.gif')     # 1-bit
compare_cover_stego('lena.gif', 'lena_stego_2bit.gif') # 2-bit


# # TEST: 2-bit mode round-trip (baboon.gif)
# result = embed_2bit('msg.txt', 'baboon.gif', 'baboon_stego_2bit.gif')
# assert result == True
# extracted = extract_2bit('baboon_stego_2bit.gif', 'extr_msg_2bit.txt')
# with open('msg.txt', 'r') as f:
#     original = f.read()
# assert extracted == original
# print("2-bit mode round-trip: PASSED")

# # Compare visual distortion: 1-bit vs 2-bit
# compare_cover_stego('baboon.gif', 'baboon_stego.gif')     # 1-bit
# compare_cover_stego('baboon.gif', 'baboon_stego_2bit.gif') # 2-bit

### Reflection: 1-Bit vs 2-Bit Tradeoffs

After implementing `embed_2bit`, compare the two modes experimentally:
- How does the percentage of changed pixels differ?
- What is the average swap distance for 1-bit vs 2-bit mode?
- Is the visual difference perceptible on `lena.gif`? On `baboon.gif`?

YOUR ANSWER HERE

- **How does the percentage of changed pixels differ?**
    - In 1-bit mode, each pixel represents 1 of 2 states, meaning the probability of a match is 50%. Thus, the percentage of changed pixels is approximately 50% of the payload capacity.
    - In 2-bit mode, each pixel represents 1 of 4 states, dropping the probability of a match to 25%. Thus, the percentage of pixels forced to swap colors increases significantly, averaging around 75%.
- **What is the average swap distance for 1-bit vs 2-bit mode?**
    - In 2-bit mode, there are 4 parity groups instead of 2. The nearest entry with a specific 2-bit value may be farther away than the nearest entry with just the opposite 1-bit parity. The algorithm is frequently forced to choose replacement colors that are further away in the RGB color space, leading to higher distortion per swapped pixel.
- **Is the visual difference perceptible on `lena.gif`? On `baboon.gif`?**
    - `lena.gif`: Visual artifacts and color distortions in 2-bit mode are extremely pronounced and generally unacceptable (PSNR_1bit=28.94dB vs. PSNR_2bit=25.21dB), as smooth pixels are forced to adopt heavily deviated colors.
    - `baboon.gif`: The visual difference between 1-bit and 2-bit is virtually imperceptible, as shown by their very close PSNR values (PSNR_1bit=32.83 dB vs. PSNR_2bit=32.14 dB). Unlike smooth images, `baboon.gif` contains high textural complexity with dense details (fur, whiskers) that effectively hide the color deviations from the 2-bit embedding, making the noise invisible to the human eye and maintaining a harmonious appearance.